# 📊 FYP Data Preparation & Cleaning
## A Prescriptive Data Analytics Dashboard on Spending Behaviour Amongst Malaysian Fresh Graduates to Empower Financial Awareness
### Husna Binti Zakaria | Universiti Teknologi MARA | January 2026

---

**Project Framework:** Adapted CRISP-DM (Phase 2: Data Preparation)  
**Objectives Supported:** Objective 1 — Analyse spending behaviour patterns using descriptive and diagnostic analytics  
**Datasets:**
- 📋 **Primary Data:** Survey responses from Malaysian fresh graduates (Google Forms)
- 📂 **Secondary Data:** DOSM Graduates Statistics 2024 (Department of Statistics Malaysia)

**Output Datasets for Power BI Dashboard:**
1. `PRIMARY_Survey_Cleaned.csv` — Cleaned survey responses with 50/30/20 logic applied
2. `DOSM_Salary_Benchmarks.csv` — National salary benchmarks by age group & qualification
3. `DOSM_Graduate_Employment.csv` — National employment statistics by state & year
4. `DOSM_State_Graduates.csv` — State-level graduate statistics
5. `PRESCRIPTIVE_Alerts.csv` — Rule-based prescriptive alert flags per respondent
6. `HYBRID_Dashboard_Master.csv` — Combined master table for Power BI relationships


---
## ⚙️ CELL 1: Install & Import Libraries

In [ ]:
# Install required libraries (run once in Colab)
!pip install openpyxl xlrd --quiet

import pandas as pd
import numpy as np
import re
import warnings
from google.colab import files
import io

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

print('✅ All libraries loaded successfully.')
print('📌 Project: Prescriptive Data Analytics Dashboard — FYP Husna Zakaria')
print('📌 Phase:   CRISP-DM Phase 2 — Data Preparation')

✅ All libraries loaded successfully.
📌 Project: Prescriptive Data Analytics Dashboard — FYP Husna Zakaria
📌 Phase:   CRISP-DM Phase 2 — Data Preparation


---
## 📤 CELL 2: Upload Raw Data Files
> Upload both files:
> 1. `FYP_Questionnaire_...Form_Responses_1.csv` (Survey primary data)
> 2. `2__Jadual_Penerbitan_Statistik_Siswazah_2024.xlsx` (DOSM secondary data)

In [ ]:
# ── Upload files from local machine ──────────────────────────────────────
print('📂 Please upload your two raw data files...')
uploaded = files.upload()

# Identify file names dynamically
csv_file  = [k for k in uploaded.keys() if k.endswith('.csv')][0]
xlsx_file = [k for k in uploaded.keys() if k.endswith('.xlsx')][0]

print(f'\n✅ Survey file   detected : {csv_file}')
print(f'✅ DOSM XLSX file detected : {xlsx_file}')

📂 Please upload your two raw data files...


Saving 2. Jadual Penerbitan Statistik Siswazah 2024.xlsx to 2. Jadual Penerbitan Statistik Siswazah 2024.xlsx
Saving FYP Questionnaire_ Spending Behaviour of Malaysian Fresh Graduates Response - Form Responses 1.csv to FYP Questionnaire_ Spending Behaviour of Malaysian Fresh Graduates Response - Form Responses 1.csv

✅ Survey file   detected : FYP Questionnaire_ Spending Behaviour of Malaysian Fresh Graduates Response - Form Responses 1.csv
✅ DOSM XLSX file detected : 2. Jadual Penerbitan Statistik Siswazah 2024.xlsx


---
## 🔍 CELL 3: Load & Inspect Raw Survey Data (Quality Check)

In [ ]:
# ── Load raw CSV ──────────────────────────────────────────────────────────────
df_raw = pd.read_csv(io.BytesIO(uploaded[csv_file]))

print('=' * 65)
print('  RAW SURVEY DATA — INITIAL QUALITY REPORT')
print('=' * 65)
print(f'  Total rows (responses)   : {len(df_raw)}')
print(f'  Total columns            : {len(df_raw.columns)}')
print(f'  Duplicate rows           : {df_raw.duplicated().sum()}')
print(f'  Total missing values     : {df_raw.isnull().sum().sum()}')
print('=' * 65)
print()

# Show missing values per column
missing = df_raw.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print('🔴 Columns with missing values:')
    for col, cnt in missing.items():
        print(f'   {col[:70]} → {cnt} missing')
else:
    print('✅ No missing values detected.')

print()
print('📋 Column names (raw):')
for i, c in enumerate(df_raw.columns):
    print(f'  [{i:02d}] {c[:90]}')

  RAW SURVEY DATA — INITIAL QUALITY REPORT
  Total rows (responses)   : 102
  Total columns            : 21
  Duplicate rows           : 0
  Total missing values     : 41

🔴 Columns with missing values:
   Column 1 → 1 missing
     1. Do you agree to participate in this survey based on the formal te → 2 missing
   2. Age / Umur → 2 missing
     3. Gender / Jantina → 2 missing
   4.  Highest Level of Education / Tahap Pendidikan Tertinggi  → 2 missing
   5. State of current residence / Negeri tempat tinggal sekarang  → 2 missing
   6. Which area best describes your current residence? / Apakah kategori → 2 missing
     7. Current Employment Status / Status Pekerjaan Semasa → 2 missing
     8. Estimated Monthly Net Income (After EPF, SOCSO, Tax) / Anggaran P → 2 missing
     9. I keep a close watch on my personal financial affairs. / Saya mem → 2 missing
     10. Before I buy something, I carefully consider whether I can affor → 2 missing
     11. I pay my bills on time and rarely incur l

---
## 🧹 CELL 4: Rename Columns to Clean Short Names (ETL Step)

In [ ]:
# ── Rename all 21 columns to clean, Power BI-friendly names ──────────────────
# Original columns are bilingual and contain emojis — strip all that out.

column_rename_map = {
    df_raw.columns[0]:  'Timestamp',
    df_raw.columns[1]:  'Consent',
    df_raw.columns[2]:  'Age_Group',
    df_raw.columns[3]:  'Gender',
    df_raw.columns[4]:  'Education_Level',
    df_raw.columns[5]:  'State',
    df_raw.columns[6]:  'Area_Type',
    df_raw.columns[7]:  'Employment_Status',
    df_raw.columns[8]:  'Monthly_Income_Band',
    df_raw.columns[9]:  'Q9_Financial_Watchfulness',
    df_raw.columns[10]: 'Q10_Considers_Affordability',
    df_raw.columns[11]: 'Q11_Pays_Bills_OnTime',
    df_raw.columns[12]: 'Q12_Sets_LongTerm_Goals',
    df_raw.columns[13]: 'Q13_Income_Barely_Lasts',
    df_raw.columns[14]: 'Needs_Spend_Band',
    df_raw.columns[15]: 'Wants_Spend_Band',
    df_raw.columns[16]: 'Savings_Debt_Band',
    df_raw.columns[17]: 'Savings_Barrier',
    df_raw.columns[18]: 'Spending_Tracking_Method',
    df_raw.columns[19]: 'Preferred_Dashboard_Feature',
    df_raw.columns[20]: 'Likelihood_Behaviour_Change',
}

df = df_raw.rename(columns=column_rename_map).copy()

print('✅ Columns renamed to clean Power BI-friendly names.')
print(f'   Columns: {list(df.columns)}')

✅ Columns renamed to clean Power BI-friendly names.
   Columns: ['Timestamp', 'Consent', 'Age_Group', 'Gender', 'Education_Level', 'State', 'Area_Type', 'Employment_Status', 'Monthly_Income_Band', 'Q9_Financial_Watchfulness', 'Q10_Considers_Affordability', 'Q11_Pays_Bills_OnTime', 'Q12_Sets_LongTerm_Goals', 'Q13_Income_Barely_Lasts', 'Needs_Spend_Band', 'Wants_Spend_Band', 'Savings_Debt_Band', 'Savings_Barrier', 'Spending_Tracking_Method', 'Preferred_Dashboard_Feature', 'Likelihood_Behaviour_Change']


---
## 🧹 CELL 5: Data Cleaning — Remove Noise & Standardise Values

In [ ]:
# ── Step 1: Drop consent-rejected rows & duplicates ──────────────────────────
initial_count = len(df)

# Keep only agreed respondents
df = df[df['Consent'].astype(str).str.contains('Yes|Setuju|Bersetuju', case=False, na=False)]
after_consent = len(df)

# Drop full duplicates
df = df.drop_duplicates()
after_dedup = len(df)

# Reset index
df = df.reset_index(drop=True)
df['Respondent_ID'] = ['R' + str(i+1).zfill(3) for i in range(len(df))]

print(f'📊 Rows before cleaning : {initial_count}')
print(f'📊 After consent filter : {after_consent}')
print(f'📊 After deduplication  : {after_dedup}')
print()

# ── Step 2: Strip emojis & bilingual Malay text from categorical values ───────
def clean_category(val):
    """Remove emojis and Malay bilingual text (after '/' or after Malay portion)."""
    if pd.isna(val):
        return np.nan
    val = str(val)
    # Remove emojis (unicode ranges)
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F9FF"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001F900-\U0001F9FF"
        u"\U0001FA00-\U0001FA6F"
        u"\U0001FA70-\U0001FAFF"
        u"\u2600-\u26FF"
        u"\u2700-\u27BF"
        "]+", flags=re.UNICODE)
    val = emoji_pattern.sub('', val)
    # Take English part before '/' if bilingual
    if ' / ' in val:
        val = val.split(' / ')[0]
    return val.strip()

categorical_cols = [
    'Age_Group', 'Gender', 'Education_Level', 'State', 'Area_Type',
    'Employment_Status', 'Monthly_Income_Band', 'Needs_Spend_Band',
    'Wants_Spend_Band', 'Savings_Debt_Band', 'Spending_Tracking_Method',
    'Likelihood_Behaviour_Change'
]

for col in categorical_cols:
    df[col] = df[col].apply(clean_category)

print('✅ Emojis and Malay bilingual text stripped from categorical columns.')

NameError: name 'df' is not defined

---
## 🔁 CELL 6: Standardise All Categorical Values

In [ ]:
# ── Standardise Gender ────────────────────────────────────────────────────────
gender_map = {
    'Female': 'Female',
    'Male': 'Male',
    'Perempuan': 'Female',
    'Lelaki': 'Male',
}
df['Gender'] = df['Gender'].replace(gender_map)

# ── Standardise Age Group ─────────────────────────────────────────────────────
df['Age_Group'] = df['Age_Group'].str.replace(r'24 years old and below.*', '24 and Below', regex=True)
df['Age_Group'] = df['Age_Group'].str.replace(r'25.*34.*', '25 - 34', regex=True)
df['Age_Group'] = df['Age_Group'].str.replace(r'35 years old and above.*', '35 and Above', regex=True)

# ── Standardise Area ─────────────────────────────────────────────────────────
df['Area_Type'] = df['Area_Type'].str.replace(r'Urban.*', 'Urban', regex=True)
df['Area_Type'] = df['Area_Type'].str.replace(r'Rural.*', 'Rural', regex=True)

# ── Standardise Employment Status ────────────────────────────────────────────
emp_map = {
    'Pursuing further studies': 'Further Studies',
    'Employed Full-Time': 'Full-Time Employed',
    'Employed Part-Time': 'Part-Time Employed',
    'Currently seeking employment': 'Job Seeking',
}
for old, new in emp_map.items():
    df['Employment_Status'] = df['Employment_Status'].str.replace(old, new, regex=False)
df['Employment_Status'] = df['Employment_Status'].apply(
    lambda x: clean_category(x) if pd.notna(x) else x
)

# ── Standardise Income Band ───────────────────────────────────────────────────
income_map = {
    'Less than RM 1,500':     'Below RM 1,500',
    'RM 1,500 - RM 2,500':    'RM 1,500 - RM 2,500',
    'RM 2,501 - RM 3,500':    'RM 2,501 - RM 3,500',
    'RM 3,501 - RM 4,500':    'RM 3,501 - RM 4,500',
    'Above RM 4,500':         'Above RM 4,500',
}
for old, new in income_map.items():
    df['Monthly_Income_Band'] = df['Monthly_Income_Band'].str.replace(old, new, regex=False)
df['Monthly_Income_Band'] = df['Monthly_Income_Band'].apply(
    lambda x: x.replace('💵', '').replace('📉', '').replace('📈', '').strip() if pd.notna(x) else x
)

# ── Standardise Spending Bands (Needs / Wants / Savings) ─────────────────────
def clean_rm_band(val):
    if pd.isna(val): return np.nan
    val = str(val)
    val = re.sub(r'Less than (RM [0-9,]+).*', r'Below \1', val)
    val = re.sub(r'(RM [0-9,]+ and above).*', r'\1', val)
    val = re.sub(r'(RM 0 \(Unable to save\)).*', 'RM 0 (No Savings)', val)
    return val.strip()

df['Needs_Spend_Band']  = df['Needs_Spend_Band'].apply(clean_rm_band)
df['Wants_Spend_Band']  = df['Wants_Spend_Band'].apply(clean_rm_band)
df['Savings_Debt_Band'] = df['Savings_Debt_Band'].apply(clean_rm_band)

# ── Standardise Tracking Method ───────────────────────────────────────────────
tracking_map = {
    'I only review bank statements': 'Bank Statements Only',
    'Mobile': 'Mobile/Web App',
    'Digital Spreadsheet': 'Digital Spreadsheet',
    'Physical Notes': 'Physical Notes/Journal',
    'I do not track': 'No Tracking',
}
for key, val in tracking_map.items():
    df['Spending_Tracking_Method'] = df['Spending_Tracking_Method'].str.replace(
        f'.*{key}.*', val, regex=True
    )

# ── Standardise Behaviour Change Likelihood ───────────────────────────────────
likelihood_map = {
    'Very Likely':   'Very Likely',
    'Likely':        'Likely',
    'Neutral':       'Neutral',
    'Unlikely':      'Unlikely',
    'Very Unlikely': 'Very Unlikely',
}
for key in likelihood_map:
    df['Likelihood_Behaviour_Change'] = df['Likelihood_Behaviour_Change'].str.replace(
        f'.*{key}.*', key, regex=True
    )

print('✅ All categorical columns standardised.')
print()
print('📊 Value counts check — Gender:')
print(df['Gender'].value_counts())
print()
print('📊 Value counts check — Age Group:')
print(df['Age_Group'].value_counts())

✅ All categorical columns standardised.

📊 Value counts check — Gender:
Gender
Female    55
Male      45
Name: count, dtype: int64

📊 Value counts check — Age Group:
Age_Group
24 and Below    69
35 and Above    17
25 - 34         14
Name: count, dtype: int64


---
## 📐 CELL 7: Numeric Midpoint Encoding for Income & Spending Bands
> Required for 50/30/20 prescriptive logic calculations in Power BI DAX

In [ ]:
# ── Map income bands to numeric midpoints (RM) ───────────────────────────────
# Midpoints are used for 50/30/20 threshold calculations in Power BI.
# These midpoints represent the central estimate for each band.

INCOME_MIDPOINTS = {
    'Below RM 1,500':      1000,   # Conservative estimate for unemployed/students
    'RM 1,500 - RM 2,500': 2000,   # Midpoint
    'RM 2,501 - RM 3,500': 3000,   # Midpoint
    'RM 3,501 - RM 4,500': 4000,   # Midpoint
    'Above RM 4,500':      5000,   # Conservative ceiling estimate
}

NEEDS_MIDPOINTS = {
    'Below RM 500':        250,
    'RM 500 - RM 999':     750,
    'RM 1,000 - RM 1,499': 1250,
    'RM 1,500 - RM 1,999': 1750,
    'RM 2,000 and above':  2200,
}

WANTS_MIDPOINTS = {
    'Below RM 300':        150,
    'RM 300 - RM 599':     450,
    'RM 600 - RM 899':     750,
    'RM 900 - RM 1,199':   1050,
    'RM 1,200 and above':  1400,
}

SAVINGS_MIDPOINTS = {
    'RM 0 (No Savings)':   0,
    'RM 1 - RM 299':       150,
    'RM 300 - RM 599':     450,
    'RM 600 - RM 899':     750,
    'RM 900 and above':    1050,
}

# Apply midpoint mappings
df['Income_Midpoint_RM'] = df['Monthly_Income_Band'].map(INCOME_MIDPOINTS)
df['Needs_Midpoint_RM']  = df['Needs_Spend_Band'].map(NEEDS_MIDPOINTS)
df['Wants_Midpoint_RM']  = df['Wants_Spend_Band'].map(WANTS_MIDPOINTS)
df['Savings_Midpoint_RM']= df['Savings_Debt_Band'].map(SAVINGS_MIDPOINTS)

# Fill any unmatched bands with NaN and report
for col in ['Income_Midpoint_RM', 'Needs_Midpoint_RM', 'Wants_Midpoint_RM', 'Savings_Midpoint_RM']:
    n_missing = df[col].isna().sum()
    if n_missing > 0:
        print(f'⚠️  {col}: {n_missing} unmatched → filling with column median')
        df[col] = df[col].fillna(df[col].median())

print('✅ Numeric midpoints encoded for all spending and income bands.')
print()
print(df[['Monthly_Income_Band','Income_Midpoint_RM',
          'Needs_Spend_Band','Needs_Midpoint_RM',
          'Wants_Spend_Band','Wants_Midpoint_RM',
          'Savings_Debt_Band','Savings_Midpoint_RM']].head(5))

⚠️  Savings_Midpoint_RM: 5 unmatched → filling with column median
✅ Numeric midpoints encoded for all spending and income bands.

   Monthly_Income_Band  Income_Midpoint_RM Needs_Spend_Band  \
0       Below RM 1,500                1000     Below RM 500   
1       Below RM 1,500                1000     Below RM 500   
2  RM 1,500 - RM 2,500                2000  RM 500 - RM 999   
3       Below RM 1,500                1000     Below RM 500   
4  RM 1,500 - RM 2,500                2000  RM 500 - RM 999   

   Needs_Midpoint_RM Wants_Spend_Band  Wants_Midpoint_RM Savings_Debt_Band  \
0                250     Below RM 300                150     RM 1 - RM 299   
1                250     Below RM 300                150     RM 1 - RM 299   
2                750  RM 300 - RM 599                450  RM 900 and above   
3                250     Below RM 300                150     RM 1 - RM 299   
4                750     Below RM 300                150   RM 300 - RM 599   

   Savings_Midpoint_RM

---
## 🎯 CELL 8: Apply 50/30/20 Rule — Core Prescriptive Analytics Logic
> This is the heart of the FYP — deterministic If-Then rule-based prescriptive engine

In [ ]:
# ── 50/30/20 Budget Thresholds ────────────────────────────────────────────────
# Rule: 50% Needs | 30% Wants | 20% Savings
# Reference: Ahmad & Mohamed Zabri (2023); Investopedia (2025)

df['Budget_Needs_Limit_RM']    = (df['Income_Midpoint_RM'] * 0.50).round(2)
df['Budget_Wants_Limit_RM']    = (df['Income_Midpoint_RM'] * 0.30).round(2)
df['Budget_Savings_Target_RM'] = (df['Income_Midpoint_RM'] * 0.20).round(2)

# ── Actual vs Recommended Spending Gap ───────────────────────────────────────
df['Needs_Gap_RM']   = (df['Needs_Midpoint_RM']   - df['Budget_Needs_Limit_RM']).round(2)
df['Wants_Gap_RM']   = (df['Wants_Midpoint_RM']   - df['Budget_Wants_Limit_RM']).round(2)
df['Savings_Gap_RM'] = (df['Budget_Savings_Target_RM'] - df['Savings_Midpoint_RM']).round(2)

# Positive gap = overspending; Negative = within budget

# ── Actual Spending Percentage of Income ─────────────────────────────────────
df['Needs_Pct_of_Income']   = ((df['Needs_Midpoint_RM']  / df['Income_Midpoint_RM']) * 100).round(1)
df['Wants_Pct_of_Income']   = ((df['Wants_Midpoint_RM']  / df['Income_Midpoint_RM']) * 100).round(1)
df['Savings_Pct_of_Income'] = ((df['Savings_Midpoint_RM']/ df['Income_Midpoint_RM']) * 100).round(1)

# ── DSR: Debt Service Ratio (Total commitments / Income) ─────────────────────
# Bank Negara Malaysia: Healthy DSR threshold = 40% or below
df['Total_Commitments_RM'] = df['Needs_Midpoint_RM'] + df['Wants_Midpoint_RM']
df['DSR_Pct']              = ((df['Total_Commitments_RM'] / df['Income_Midpoint_RM']) * 100).round(1)
df['DSR_Status']           = df['DSR_Pct'].apply(lambda x: 'Healthy' if x <= 40 else
                                                  ('Warning' if x <= 60 else 'Critical'))

print('✅ 50/30/20 thresholds and gaps calculated.')
print()
print('Sample — Budget vs Actual (first 5 respondents):')
print(df[['Respondent_ID',
          'Income_Midpoint_RM',
          'Budget_Needs_Limit_RM', 'Needs_Midpoint_RM', 'Needs_Gap_RM',
          'Budget_Wants_Limit_RM', 'Wants_Midpoint_RM', 'Wants_Gap_RM',
          'Budget_Savings_Target_RM', 'Savings_Midpoint_RM', 'Savings_Gap_RM',
          'DSR_Pct', 'DSR_Status']].head())

✅ 50/30/20 thresholds and gaps calculated.

Sample — Budget vs Actual (first 5 respondents):
  Respondent_ID  Income_Midpoint_RM  Budget_Needs_Limit_RM  Needs_Midpoint_RM  \
0          R001                1000                  500.0                250   
1          R002                1000                  500.0                250   
2          R003                2000                 1000.0                750   
3          R004                1000                  500.0                250   
4          R005                2000                 1000.0                750   

   Needs_Gap_RM  Budget_Wants_Limit_RM  Wants_Midpoint_RM  Wants_Gap_RM  \
0        -250.0                  300.0                150        -150.0   
1        -250.0                  300.0                150        -150.0   
2        -250.0                  600.0                450        -150.0   
3        -250.0                  300.0                150        -150.0   
4        -250.0                  600.0       

---
## 🚦 CELL 9: Generate Prescriptive Alert Flags (If-Then Rules)
> Rule-Based Heuristics aligned with FYP Section 3.5.2 Process Flow

In [ ]:
# ── RULE 1: Needs Overspend Alert ─────────────────────────────────────────────
# IF Needs > 50% of income → Alert: "Essential spending exceeds safe limit"
df['Alert_Needs_Overspend'] = (df['Needs_Midpoint_RM'] > df['Budget_Needs_Limit_RM']).astype(int)

# ── RULE 2: Wants (Lifestyle Inflation) Alert ─────────────────────────────────
# IF Wants > 30% of income → Alert: "Lifestyle Inflation Detected"
df['Alert_Wants_Overspend'] = (df['Wants_Midpoint_RM'] > df['Budget_Wants_Limit_RM']).astype(int)

# ── RULE 3: Insufficient Savings Alert ───────────────────────────────────────
# IF Savings < 20% of income → Alert: "Savings below 20% target"
df['Alert_Savings_Deficit'] = (df['Savings_Midpoint_RM'] < df['Budget_Savings_Target_RM']).astype(int)

# ── RULE 4: Zero Savings Critical Alert ──────────────────────────────────────
# IF Savings = 0 → Critical: "No savings — high debt risk"
df['Alert_Zero_Savings'] = (df['Savings_Midpoint_RM'] == 0).astype(int)

# ── RULE 5: DSR Danger Alert ──────────────────────────────────────────────────
# IF DSR > 60% → Critical financial health
df['Alert_DSR_Danger'] = (df['DSR_Pct'] > 60).astype(int)

# ── Overall Financial Health Score (RAG — Red/Amber/Green) ────────────────────
# Green  (0 alerts)       = Financially On Track
# Amber  (1-2 alerts)     = Warning Zone
# Red    (3+ alerts)      = Danger Zone
df['Alert_Total_Count'] = (
    df['Alert_Needs_Overspend'] +
    df['Alert_Wants_Overspend'] +
    df['Alert_Savings_Deficit'] +
    df['Alert_Zero_Savings']    +
    df['Alert_DSR_Danger']
)

def rag_status(count):
    if count == 0:    return 'Green — On Track'
    elif count <= 2:  return 'Amber — Warning'
    else:             return 'Red — Danger'

df['RAG_Financial_Status'] = df['Alert_Total_Count'].apply(rag_status)

# ── Prescriptive Advice Text (for Power BI Smart Narrative cards) ─────────────
def generate_advice(row):
    advice = []
    if row['Alert_Zero_Savings']:
        advice.append('⛔ CRITICAL: You are saving RM 0. Start an emergency fund immediately — target RM ' +
                      str(int(row['Budget_Savings_Target_RM'])) + '/month.')
    elif row['Alert_Savings_Deficit']:
        shortfall = round(row['Savings_Gap_RM'], 0)
        advice.append(f'⚠️ SAVINGS: You need RM {int(shortfall)} more/month to reach the 20% savings goal.')
    if row['Alert_Wants_Overspend']:
        excess = round(row['Wants_Gap_RM'], 0)
        advice.append(f'⚠️ LIFESTYLE: Reduce wants spending by RM {int(excess)}/month to stay within 30%.')
    if row['Alert_Needs_Overspend']:
        advice.append('⚠️ NEEDS: Essential spending exceeds 50%. Review fixed costs like rent or transport.')
    if row['Alert_DSR_Danger']:
        advice.append(f'🚨 DSR ALERT: Your debt-service ratio is {row["DSR_Pct"]}% — seek credit counselling (AKPK).')
    if not advice:
        advice.append('✅ GREAT JOB! Your spending aligns with the 50/30/20 rule. Keep it up!')
    return ' | '.join(advice)

df['Prescriptive_Advice'] = df.apply(generate_advice, axis=1)

print('✅ Prescriptive alert flags generated.')
print()
print('📊 RAG Financial Status Distribution:')
print(df['RAG_Financial_Status'].value_counts())
print()
print('📊 Wants Overspend Alert (1=Yes, 0=No):')
print(df['Alert_Wants_Overspend'].value_counts())

✅ Prescriptive alert flags generated.

📊 RAG Financial Status Distribution:
RAG_Financial_Status
Amber — Warning     51
Green — On Track    29
Red — Danger        20
Name: count, dtype: int64

📊 Wants Overspend Alert (1=Yes, 0=No):
Alert_Wants_Overspend
0    82
1    18
Name: count, dtype: int64


---
## 📊 CELL 10: Financial Literacy Score Computation
> Questions 9–13 are 1–5 Likert scale items measuring financial awareness

In [ ]:
# ── Financial Awareness Score (Q9 to Q13) ────────────────────────────────────
# Q9-Q12: Higher = more aware (positive indicators)
# Q13: Reversed — higher = LESS capable (income barely lasts)

likert_cols = [
    'Q9_Financial_Watchfulness',
    'Q10_Considers_Affordability',
    'Q11_Pays_Bills_OnTime',
    'Q12_Sets_LongTerm_Goals',
    'Q13_Income_Barely_Lasts',   # Reverse-coded
]

# Ensure numeric
for col in likert_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Reverse-code Q13 (1→5, 2→4, 3→3, 4→2, 5→1)
df['Q13_Reversed'] = 6 - df['Q13_Income_Barely_Lasts']

# Mean of all 5 (after reverse-coding Q13)
score_cols = ['Q9_Financial_Watchfulness', 'Q10_Considers_Affordability',
              'Q11_Pays_Bills_OnTime', 'Q12_Sets_LongTerm_Goals', 'Q13_Reversed']
df['Financial_Awareness_Score'] = df[score_cols].mean(axis=1).round(2)

# Categorise awareness level
def awareness_level(score):
    if score >= 4.0:  return 'High Awareness'
    elif score >= 3.0: return 'Moderate Awareness'
    else:              return 'Low Awareness'

df['Awareness_Level'] = df['Financial_Awareness_Score'].apply(awareness_level)

# Knowledge-Action Gap Flag
# High awareness BUT still has savings deficit = Knowledge-Action Gap
df['KA_Gap_Flag'] = (
    (df['Awareness_Level'] == 'High Awareness') &
    (df['Alert_Savings_Deficit'] == 1)
).astype(int)

print('✅ Financial Awareness Score & Knowledge-Action Gap flag computed.')
print()
print('📊 Awareness Level Distribution:')
print(df['Awareness_Level'].value_counts())
print()
print('📊 Knowledge-Action Gap (1=Gap Exists, 0=No Gap):')
print(df['KA_Gap_Flag'].value_counts())
print()
print(f'   KA-Gap rate: {df["KA_Gap_Flag"].mean()*100:.1f}% of respondents show the Knowledge-Action Gap.')

✅ Financial Awareness Score & Knowledge-Action Gap flag computed.

📊 Awareness Level Distribution:
Awareness_Level
High Awareness        62
Moderate Awareness    34
Low Awareness          4
Name: count, dtype: int64

📊 Knowledge-Action Gap (1=Gap Exists, 0=No Gap):
KA_Gap_Flag
0    73
1    27
Name: count, dtype: int64

   KA-Gap rate: 27.0% of respondents show the Knowledge-Action Gap.


---
## 🧹 CELL 11: Multi-Select Columns — Savings Barrier & Dashboard Feature Parsing

In [ ]:
# ── Parse multi-select barrier column (Q17) ───────────────────────────────────
# Each cell may contain multiple barriers separated by comma

BARRIER_KEYWORDS = {
    'High_Cost_of_Living':      'High cost of living',
    'Low_Salary':               'Low entry-level salary',
    'Lifestyle_Peer_Pressure':  'Overspending on lifestyle',
    'Family_Commitment':        'High financial commitment to family',
    'High_Debt':                'High existing debt',
}

for col_flag, keyword in BARRIER_KEYWORDS.items():
    df[f'Barrier_{col_flag}'] = df['Savings_Barrier'].apply(
        lambda x: 1 if pd.notna(x) and keyword.lower() in str(x).lower() else 0
    )

# ── Parse multi-select dashboard features (Q19) ───────────────────────────────
FEATURE_KEYWORDS = {
    'Feature_Wants_Warning':     'Immediate warnings when',
    'Feature_Emergency_Fund':    'build an emergency fund',
    'Feature_Lifestyle_Creep':   'Lifestyle Creep',
    'Feature_Debt_Clearance':    'debt clearance',
}

for col_flag, keyword in FEATURE_KEYWORDS.items():
    df[col_flag] = df['Preferred_Dashboard_Feature'].apply(
        lambda x: 1 if pd.notna(x) and keyword.lower() in str(x).lower() else 0
    )

print('✅ Multi-select columns parsed into binary indicator flags.')
print()
print('📊 Savings Barriers (% of respondents citing each):')
for col in [f'Barrier_{k}' for k in BARRIER_KEYWORDS]:
    pct = df[col].mean() * 100
    print(f'   {col.replace("Barrier_",""):30s}: {pct:.1f}%')

print()
print('📊 Preferred Dashboard Features (% of respondents):')
for col in FEATURE_KEYWORDS:
    pct = df[col].mean() * 100
    print(f'   {col.replace("Feature_",""):25s}: {pct:.1f}%')

✅ Multi-select columns parsed into binary indicator flags.

📊 Savings Barriers (% of respondents citing each):
   High_Cost_of_Living           : 69.0%
   Low_Salary                    : 44.0%
   Lifestyle_Peer_Pressure       : 40.0%
   Family_Commitment             : 48.0%
   High_Debt                     : 30.0%

📊 Preferred Dashboard Features (% of respondents):
   Emergency_Fund           : 40.0%
   Lifestyle_Creep          : 47.0%
   Debt_Clearance           : 33.0%


---
## ✅ CELL 12: Final Cleaned Survey Dataset — Select & Export

In [ ]:
# ── Select final columns for Power BI ────────────────────────────────────────
survey_final_cols = [
    # Identifiers
    'Respondent_ID',
    # Demographics
    'Age_Group', 'Gender', 'Education_Level', 'State', 'Area_Type',
    # Financial profile
    'Employment_Status', 'Monthly_Income_Band', 'Income_Midpoint_RM',
    # Spending bands (raw)
    'Needs_Spend_Band', 'Wants_Spend_Band', 'Savings_Debt_Band',
    # Spending numeric midpoints
    'Needs_Midpoint_RM', 'Wants_Midpoint_RM', 'Savings_Midpoint_RM',
    # 50/30/20 Budget limits
    'Budget_Needs_Limit_RM', 'Budget_Wants_Limit_RM', 'Budget_Savings_Target_RM',
    # Gaps
    'Needs_Gap_RM', 'Wants_Gap_RM', 'Savings_Gap_RM',
    # % of income
    'Needs_Pct_of_Income', 'Wants_Pct_of_Income', 'Savings_Pct_of_Income',
    # DSR
    'Total_Commitments_RM', 'DSR_Pct', 'DSR_Status',
    # Prescriptive alerts
    'Alert_Needs_Overspend', 'Alert_Wants_Overspend',
    'Alert_Savings_Deficit', 'Alert_Zero_Savings', 'Alert_DSR_Danger',
    'Alert_Total_Count', 'RAG_Financial_Status',
    # Advice text
    'Prescriptive_Advice',
    # Financial awareness
    'Q9_Financial_Watchfulness', 'Q10_Considers_Affordability',
    'Q11_Pays_Bills_OnTime', 'Q12_Sets_LongTerm_Goals', 'Q13_Income_Barely_Lasts',
    'Financial_Awareness_Score', 'Awareness_Level', 'KA_Gap_Flag',
    # Barriers (binary)
    'Barrier_High_Cost_of_Living', 'Barrier_Low_Salary',
    'Barrier_Lifestyle_Peer_Pressure', 'Barrier_Family_Commitment', 'Barrier_High_Debt',
    # Dashboard feature preferences (binary)
    'Feature_Wants_Warning', 'Feature_Emergency_Fund',
    'Feature_Lifestyle_Creep', 'Feature_Debt_Clearance',
    # Behaviour change likelihood
    'Likelihood_Behaviour_Change', 'Spending_Tracking_Method',
]

df_survey_clean = df[survey_final_cols].copy()

# Final null check
null_report = df_survey_clean.isnull().sum()
null_report = null_report[null_report > 0]
if len(null_report) > 0:
    print('⚠️ Remaining nulls (filled with mode/median):')
    for col, cnt in null_report.items():
        if df_survey_clean[col].dtype == 'object':
            fill_val = df_survey_clean[col].mode()[0]
        else:
            fill_val = df_survey_clean[col].median()
        df_survey_clean[col] = df_survey_clean[col].fillna(fill_val)
        print(f'   {col}: {cnt} nulls → filled with {fill_val}')
else:
    print('✅ No nulls remaining in final survey dataset.')

print(f'\n📊 FINAL SURVEY DATASET: {df_survey_clean.shape[0]} rows × {df_survey_clean.shape[1]} columns')

# Export
df_survey_clean.to_csv('PRIMARY_Survey_Cleaned.csv', index=False, encoding='utf-8-sig')
print('\n💾 Saved: PRIMARY_Survey_Cleaned.csv')
print(df_survey_clean.head(3))

✅ No nulls remaining in final survey dataset.

📊 FINAL SURVEY DATASET: 100 rows × 54 columns

💾 Saved: PRIMARY_Survey_Cleaned.csv
  Respondent_ID     Age_Group  Gender Education_Level         State Area_Type  \
0          R001  24 and Below  Female          Degree         Kedah     Urban   
1          R002  24 and Below  Female          Degree      Kelantan     Urban   
2          R003  24 and Below  Female          Degree  Pulau Pinang     Urban   

    Employment_Status  Monthly_Income_Band  Income_Midpoint_RM  \
0     Further Studies       Below RM 1,500                1000   
1     Further Studies       Below RM 1,500                1000   
2  Full-Time Employed  RM 1,500 - RM 2,500                2000   

  Needs_Spend_Band Wants_Spend_Band Savings_Debt_Band  Needs_Midpoint_RM  \
0     Below RM 500     Below RM 300     RM 1 - RM 299                250   
1     Below RM 500     Below RM 300     RM 1 - RM 299                250   
2  RM 500 - RM 999  RM 300 - RM 599  RM 900 and abov

---
## 🏛️ CELL 13: Load & Parse DOSM Secondary Data (Salary Benchmarks)

In [ ]:
# ── Load DOSM XLSX ────────────────────────────────────────────────────────────
dosm_bytes = io.BytesIO(uploaded[xlsx_file])

# ── Table 10a: Mean Monthly Salary by Age Group (2020-2024) ──────────────────
dosm_salary_mean = pd.DataFrame({
    'Year':             [2020, 2021, 2022, 2023, 2024],
    'Total_Mean_RM':    [4466,  4542,  4747,  4933,  5330],
    'Age_24Below_Mean': [1951,  1976,  2088,  2242,  2456],
    'Age_25_34_Mean':   [3413,  3522,  3580,  3820,  4126],
    'Age_35_44_Mean':   [5358,  5362,  5475,  5663,  5970],
    'Age_45Plus_Mean':  [6914,  6987,  7159,  7442,  7660],
})

# ── Table 10b: Median Monthly Salary by Age Group (2020-2024) ────────────────
dosm_salary_median = pd.DataFrame({
    'Year':               [2020, 2021, 2022, 2023, 2024],
    'Total_Median_RM':    [3711,  3956,  4265,  4409,  4521],
    'Age_24Below_Median': [1520,  1618,  1689,  1747,  2031],
    'Age_25_34_Median':   [3005,  3245,  3368,  3468,  3505],
    'Age_35_44_Median':   [4838,  5061,  5305,  5361,  5522],
    'Age_45Plus_Median':  [6189,  6538,  6975,  7050,  7191],
})

# ── Table 10e: Mean Salary by Qualification & Sex (2020-2024) ────────────────
dosm_salary_qual = pd.DataFrame({
    'Year':              [2020, 2021, 2022, 2023, 2024],
    'Total_Mean':        [4466, 4542, 4747, 4933, 5330],
    'Male_Mean':         [4910, 5004, 5199, 5541, 5816],
    'Female_Mean':       [4190, 4254, 4435, 4683, 4910],
    'Degree_Total_Mean': [5492, 5519, 5607, 5979, 6247],
    'Degree_Male_Mean':  [6291, 6323, 6334, 6514, 6949],
    'Degree_Female_Mean':[4898, 4927, 5016, 5284, 5697],
    'Diploma_Total_Mean':[3172, 3343, 3533, 3769, 4080],
    'Diploma_Male_Mean': [3327, 3518, 3771, 4201, 4451],
    'Diploma_Female_Mean':[3071,3220, 3376, 3516, 3713],
})

# ── Merge into a tidy long-format DOSM Salary Benchmarks table ───────────────
rows = []
for year in [2020, 2021, 2022, 2023, 2024]:
    yr_mean   = dosm_salary_mean[dosm_salary_mean.Year==year].iloc[0]
    yr_median = dosm_salary_median[dosm_salary_median.Year==year].iloc[0]
    yr_qual   = dosm_salary_qual[dosm_salary_qual.Year==year].iloc[0]

    # Fresh graduate benchmark (≤24) — the primary user group of this FYP
    rows.append({
        'Year':               year,
        'Age_Group':          '24 and Below',
        'Qualification':      'All',
        'Gender':             'All',
        'Mean_Salary_RM':     int(yr_mean['Age_24Below_Mean']),
        'Median_Salary_RM':   int(yr_median['Age_24Below_Median']),
        'DOSM_Needs_50pct':   round(int(yr_mean['Age_24Below_Mean']) * 0.50, 0),
        'DOSM_Wants_30pct':   round(int(yr_mean['Age_24Below_Mean']) * 0.30, 0),
        'DOSM_Savings_20pct': round(int(yr_mean['Age_24Below_Mean']) * 0.20, 0),
    })
    # 25-34 age group
    rows.append({
        'Year':               year,
        'Age_Group':          '25 - 34',
        'Qualification':      'All',
        'Gender':             'All',
        'Mean_Salary_RM':     int(yr_mean['Age_25_34_Mean']),
        'Median_Salary_RM':   int(yr_median['Age_25_34_Median']),
        'DOSM_Needs_50pct':   round(int(yr_mean['Age_25_34_Mean']) * 0.50, 0),
        'DOSM_Wants_30pct':   round(int(yr_mean['Age_25_34_Mean']) * 0.30, 0),
        'DOSM_Savings_20pct': round(int(yr_mean['Age_25_34_Mean']) * 0.20, 0),
    })
    # Degree holders
    rows.append({
        'Year':               year,
        'Age_Group':          'All',
        'Qualification':      'Degree',
        'Gender':             'All',
        'Mean_Salary_RM':     int(yr_qual['Degree_Total_Mean']),
        'Median_Salary_RM':   int(yr_qual['Degree_Total_Mean']),
        'DOSM_Needs_50pct':   round(int(yr_qual['Degree_Total_Mean']) * 0.50, 0),
        'DOSM_Wants_30pct':   round(int(yr_qual['Degree_Total_Mean']) * 0.30, 0),
        'DOSM_Savings_20pct': round(int(yr_qual['Degree_Total_Mean']) * 0.20, 0),
    })
    # Diploma holders
    rows.append({
        'Year':               year,
        'Age_Group':          'All',
        'Qualification':      'Diploma',
        'Gender':             'All',
        'Mean_Salary_RM':     int(yr_qual['Diploma_Total_Mean']),
        'Median_Salary_RM':   int(yr_qual['Diploma_Total_Mean']),
        'DOSM_Needs_50pct':   round(int(yr_qual['Diploma_Total_Mean']) * 0.50, 0),
        'DOSM_Wants_30pct':   round(int(yr_qual['Diploma_Total_Mean']) * 0.30, 0),
        'DOSM_Savings_20pct': round(int(yr_qual['Diploma_Total_Mean']) * 0.20, 0),
    })

df_dosm_salary = pd.DataFrame(rows)
df_dosm_salary.to_csv('DOSM_Salary_Benchmarks.csv', index=False, encoding='utf-8-sig')
print(f'✅ DOSM Salary Benchmarks: {df_dosm_salary.shape[0]} rows × {df_dosm_salary.shape[1]} columns')
print('💾 Saved: DOSM_Salary_Benchmarks.csv')
print(df_dosm_salary.head(8))

✅ DOSM Salary Benchmarks: 20 rows × 9 columns
💾 Saved: DOSM_Salary_Benchmarks.csv
   Year     Age_Group Qualification Gender  Mean_Salary_RM  Median_Salary_RM  \
0  2020  24 and Below           All    All            1951              1520   
1  2020       25 - 34           All    All            3413              3005   
2  2020           All        Degree    All            5492              5492   
3  2020           All       Diploma    All            3172              3172   
4  2021  24 and Below           All    All            1976              1618   
5  2021       25 - 34           All    All            3522              3245   
6  2021           All        Degree    All            5519              5519   
7  2021           All       Diploma    All            3343              3343   

   DOSM_Needs_50pct  DOSM_Wants_30pct  DOSM_Savings_20pct  
0             976.0             585.0               390.0  
1            1706.0            1024.0               683.0  
2            2746

---
## 🏛️ CELL 14: DOSM State-Level Graduate Employment Statistics

In [ ]:
# ── State-level graduate statistics from DOSM Jadual 9a–9p ───────────────────
# Manually transcribed from DOSM Jadual 9 (all states), 2024 data
# Source: Statistik Siswazah 2024, DOSM

state_data = [
    # State, Total_Graduates_000, Labour_Force_000, Employed_000, Unemployed_000, LFPR_Pct, UR_Pct
    ('Johor',              554.9, 496.9, 480.7, 16.2, 89.6, 3.3),
    ('Kedah',              303.7, 253.0, 245.1,  7.9, 83.3, 3.1),
    ('Kelantan',           220.9, 181.5, 174.6,  6.9, 82.1, 3.8),
    ('Melaka',             183.4, 164.8, 160.3,  4.5, 89.9, 2.7),
    ('Negeri Sembilan',    184.1, 163.0, 158.4,  4.6, 88.5, 2.8),
    ('Pahang',             186.8, 161.5, 156.5,  5.0, 86.5, 3.1),
    ('Perak',              306.6, 263.8, 255.4,  8.4, 86.0, 3.2),
    ('Perlis',              42.5,  36.0,  34.9,  1.1, 84.7, 3.1),
    ('Pulau Pinang',       330.1, 296.8, 287.0,  9.8, 89.9, 3.3),
    ('Sabah',              285.6, 233.1, 222.5, 10.6, 81.6, 4.6),
    ('Sarawak',            333.1, 288.1, 277.8, 10.3, 86.5, 3.6),
    ('Selangor',           949.0, 866.4, 841.4, 25.0, 91.3, 2.9),
    ('Terengganu',         143.9, 117.6, 113.4,  4.2, 81.7, 3.6),
    ('W.P. Kuala Lumpur',  459.9, 429.1, 415.8, 13.3, 93.3, 3.1),
    ('W.P. Labuan',         16.0,  13.6,  13.1,  0.5, 85.0, 3.7),
    ('W.P. Putrajaya',      25.1,  22.6,  21.9,  0.7, 90.0, 3.1),
]

df_dosm_state = pd.DataFrame(state_data, columns=[
    'State', 'Total_Graduates_000', 'Labour_Force_000',
    'Employed_000', 'Unemployed_000',
    'LFPR_Pct', 'Unemployment_Rate_Pct'
])

df_dosm_state['Year']              = 2024
df_dosm_state['Data_Source']       = 'DOSM Statistik Siswazah 2024'
df_dosm_state['Employment_Rate_Pct'] = (100 - df_dosm_state['Unemployment_Rate_Pct']).round(1)

df_dosm_state.to_csv('DOSM_State_Graduates.csv', index=False, encoding='utf-8-sig')
print(f'✅ DOSM State Dataset: {df_dosm_state.shape[0]} rows × {df_dosm_state.shape[1]} columns')
print('💾 Saved: DOSM_State_Graduates.csv')
print(df_dosm_state)

✅ DOSM State Dataset: 16 rows × 10 columns
💾 Saved: DOSM_State_Graduates.csv
                State  Total_Graduates_000  Labour_Force_000  Employed_000  \
0               Johor                554.9             496.9         480.7   
1               Kedah                303.7             253.0         245.1   
2            Kelantan                220.9             181.5         174.6   
3              Melaka                183.4             164.8         160.3   
4     Negeri Sembilan                184.1             163.0         158.4   
5              Pahang                186.8             161.5         156.5   
6               Perak                306.6             263.8         255.4   
7              Perlis                 42.5              36.0          34.9   
8        Pulau Pinang                330.1             296.8         287.0   
9               Sabah                285.6             233.1         222.5   
10            Sarawak                333.1             288.1     

---
## 🏛️ CELL 15: DOSM National Employment Trend (2020–2024)

In [ ]:
# ── National graduate employment trend — Jadual 1b ────────────────────────────

df_dosm_national = pd.DataFrame({
    'Year':                          [2020,   2021,   2022,   2023,   2024],
    'Total_Graduates_000':           [4987.2, 5248.6, 5512.2, 5743.3, 5981.4],
    'Labour_Force_000':              [4241.5, 4463.9, 4709.2, 4923.1, 5142.6],
    'Employed_000':                  [4053.1, 4278.7, 4534.4, 4755.8, 4976.7],
    'Unemployed_000':                [188.4,  185.2,  174.8,  167.3,  165.9],
    'LFPR_Pct':                      [85.0,   85.0,   85.4,   85.7,   86.0],
    'Unemployment_Rate_Pct':         [4.4,    4.1,    3.7,    3.4,    3.2],
    # Degree-specific
    'Degree_Total_000':              [2632.8, 2813.5, 2971.4, 3121.3, 3281.3],
    'Degree_Employed_000':           [2250.6, 2404.6, 2553.2, 2697.9, 2850.9],
    'Degree_UR_Pct':                 [4.1,    4.0,    3.6,    3.3,    3.1],
    # Diploma-specific
    'Diploma_Total_000':             [2354.4, 2435.0, 2540.7, 2622.0, 2700.1],
    'Diploma_Employed_000':          [1802.4, 1874.1, 1981.2, 2057.8, 2125.8],
    'Diploma_UR_Pct':                [4.8,    4.3,    3.8,    3.6,    3.4],
    # National mean salary (from Jadual 10a)
    'National_Mean_Salary_RM':       [4466,   4542,   4747,   4933,   5330],
    'National_Median_Salary_RM':     [3711,   3956,   4265,   4409,   4521],
    # Fresh grad (≤24) mean salary
    'FreshGrad_Mean_Salary_RM':      [1951,   1976,   2088,   2242,   2456],
    'FreshGrad_Median_Salary_RM':    [1520,   1618,   1689,   1747,   2031],
    # 50/30/20 thresholds based on fresh grad mean
    'FreshGrad_Needs_Limit_RM':      [976,    988,   1044,   1121,   1228],
    'FreshGrad_Wants_Limit_RM':      [585,    593,    627,    673,    737],
    'FreshGrad_Savings_Target_RM':   [390,    395,    418,    448,    491],
    'Data_Source':                   ['DOSM']*5,
})

df_dosm_national.to_csv('DOSM_Graduate_Employment.csv', index=False, encoding='utf-8-sig')
print(f'✅ DOSM National Employment Trend: {df_dosm_national.shape[0]} rows × {df_dosm_national.shape[1]} columns')
print('💾 Saved: DOSM_Graduate_Employment.csv')
print(df_dosm_national[['Year','FreshGrad_Mean_Salary_RM',
                         'FreshGrad_Needs_Limit_RM',
                         'FreshGrad_Wants_Limit_RM',
                         'FreshGrad_Savings_Target_RM']])

✅ DOSM National Employment Trend: 5 rows × 21 columns
💾 Saved: DOSM_Graduate_Employment.csv
   Year  FreshGrad_Mean_Salary_RM  FreshGrad_Needs_Limit_RM  \
0  2020                      1951                       976   
1  2021                      1976                       988   
2  2022                      2088                      1044   
3  2023                      2242                      1121   
4  2024                      2456                      1228   

   FreshGrad_Wants_Limit_RM  FreshGrad_Savings_Target_RM  
0                       585                          390  
1                       593                          395  
2                       627                          418  
3                       673                          448  
4                       737                          491  


In [ ]:
# ============================================================
# DOSM HIES 2022/2023 — State Household Income & Expenditure
# ============================================================

# ── CELL 16b: Load & Enrich HIES State Dataset ───────────────────────────────
print('📂 Please upload: hies_state.csv')
uploaded_hies = files.upload()
hies_file = [k for k in uploaded_hies.keys() if 'hies' in k.lower()][0]

df_hies = pd.read_csv(io.BytesIO(uploaded_hies[hies_file]))
print(f'✅ Loaded: {hies_file} — {len(df_hies)} rows × {len(df_hies.columns)} columns')

# ── Rename ────────────────────────────────────────────────────────────────────
df_hies = df_hies.rename(columns={
    'date': 'Survey_Date',
    'state': 'State',
    'income_mean': 'Income_Mean_RM',
    'income_median': 'Income_Median_RM',
    'expenditure_mean': 'Expenditure_Mean_RM',
    'gini': 'Gini_Coefficient',
    'poverty': 'Poverty_Rate_Pct',
})

df_hies['Year']        = 2022
df_hies['Survey_Name'] = 'HIES 2022/2023'
df_hies['Data_Source'] = 'DOSM Household Income & Expenditure Survey (HIES) 2022/2023'

# ── 50/30/20 thresholds based on HIES state household income ─────────────────
df_hies['HIES_Needs_50pct_RM']    = (df_hies['Income_Mean_RM'] * 0.50).round(0)
df_hies['HIES_Wants_30pct_RM']    = (df_hies['Income_Mean_RM'] * 0.30).round(0)
df_hies['HIES_Savings_20pct_RM']  = (df_hies['Income_Mean_RM'] * 0.20).round(0)

# ── Expenditure pressure analytics ───────────────────────────────────────────
df_hies['Expenditure_Pct_of_Income'] = (df_hies['Expenditure_Mean_RM'] / df_hies['Income_Mean_RM'] * 100).round(1)
df_hies['Exp_vs_Needs_Gap_RM']       = (df_hies['Expenditure_Mean_RM'] - df_hies['HIES_Needs_50pct_RM']).round(0)
df_hies['Est_Monthly_Savings_RM']    = (df_hies['Income_Mean_RM'] - df_hies['Expenditure_Mean_RM']).round(0)
df_hies['Est_Savings_Pct']           = (df_hies['Est_Monthly_Savings_RM'] / df_hies['Income_Mean_RM'] * 100).round(1)

# ── National benchmark comparisons ───────────────────────────────────────────
NATIONAL_MEAN_HIES = round(df_hies['Income_Mean_RM'].mean(), 0)
df_hies['Income_vs_National_Avg_RM']  = (df_hies['Income_Mean_RM'] - NATIONAL_MEAN_HIES).round(0)
df_hies['Income_vs_National_Avg_Pct'] = ((df_hies['Income_Mean_RM'] / NATIONAL_MEAN_HIES - 1) * 100).round(1)

# ── FYP KEY METRIC: Fresh grad salary vs state median ─────────────────────────
# Fresh grad mean salary 2024 = RM 2,456 (DOSM Jadual 10a)
# This answers: "How affordable is the state for a fresh graduate?"
FRESH_GRAD_MEAN_2024 = 2456
df_hies['FreshGrad_as_Pct_StateMedian'] = round(FRESH_GRAD_MEAN_2024 / df_hies['Income_Median_RM'] * 100, 1)
df_hies['FreshGrad_Income_Gap_RM']      = (df_hies['Income_Mean_RM'] - FRESH_GRAD_MEAN_2024).round(0)
# Affordability: what % of state expenditure can a fresh grad afford?
df_hies['FreshGrad_Expenditure_Coverage_Pct'] = round(FRESH_GRAD_MEAN_2024 / df_hies['Expenditure_Mean_RM'] * 100, 1)
df_hies['FreshGrad_Affordable_State'] = (df_hies['FreshGrad_Expenditure_Coverage_Pct'] >= 70).astype(int)

# ── RAG Classifications ────────────────────────────────────────────────────────
df_hies['CoL_Pressure_Score_Pct'] = df_hies['Expenditure_Pct_of_Income']
df_hies['CoL_RAG'] = df_hies['CoL_Pressure_Score_Pct'].apply(
    lambda s: 'Critical (>70%)' if s >= 70 else ('Warning (55-70%)' if s >= 55 else 'Manageable (<55%)')
)
df_hies['Gini_RAG'] = df_hies['Gini_Coefficient'].apply(
    lambda g: 'High Inequality (≥0.38)' if g >= 0.38 else
              ('Moderate (0.35–0.38)' if g >= 0.35 else 'Low Inequality (<0.35)')
)
df_hies['Poverty_RAG'] = df_hies['Poverty_Rate_Pct'].apply(
    lambda p: 'High (≥10%)' if p >= 10 else ('Moderate (5-10%)' if p >= 5 else 'Low (<5%)')
)

# ── State rankings ─────────────────────────────────────────────────────────────
df_hies['Income_Rank']  = df_hies['Income_Mean_RM'].rank(ascending=False).astype(int)
df_hies['Poverty_Rank'] = df_hies['Poverty_Rate_Pct'].rank(ascending=False).astype(int)
df_hies['Affordability_Rank'] = df_hies['FreshGrad_Expenditure_Coverage_Pct'].rank(ascending=False).astype(int)

# ── Quality check ─────────────────────────────────────────────────────────────
assert df_hies.isnull().sum().sum() == 0, "Nulls found!"
print(f'\n✅ HIES dataset enriched: {df_hies.shape[0]} states × {df_hies.shape[1]} columns')
print(f'   Zero null values confirmed.')
print()
print('📊 KEY INSIGHTS for FYP:')
most_affordable = df_hies.loc[df_hies['FreshGrad_Expenditure_Coverage_Pct'].idxmax(), 'State']
least_affordable = df_hies.loc[df_hies['FreshGrad_Expenditure_Coverage_Pct'].idxmin(), 'State']
highest_col = df_hies.loc[df_hies['CoL_Pressure_Score_Pct'].idxmax(), 'State']
highest_poverty = df_hies.loc[df_hies['Poverty_Rate_Pct'].idxmax(), 'State']
print(f'   Most affordable for fresh grads : {most_affordable}')
print(f'   Least affordable for fresh grads: {least_affordable}')
print(f'   Highest CoL pressure            : {highest_col} ({df_hies.loc[df_hies["CoL_Pressure_Score_Pct"].idxmax(), "CoL_Pressure_Score_Pct"]}%)')
print(f'   Highest poverty rate            : {highest_poverty} ({df_hies["Poverty_Rate_Pct"].max()}%)')
print(f'   States in Critical CoL zone     : {(df_hies["CoL_RAG"]=="Critical (>70%)").sum()}')
print(f'   States affordable for fresh grads: {df_hies["FreshGrad_Affordable_State"].sum()} of 16')

# ── Save ──────────────────────────────────────────────────────────────────────
df_hies.to_csv('DOSM_HIES_State_Cleaned.csv', index=False, encoding='utf-8-sig')
files.download('DOSM_HIES_State_Cleaned.csv')
print('\n💾 Downloaded: DOSM_HIES_State_Cleaned.csv')
print()
print('📌 Add to Power BI:')
print('   Import DOSM_HIES_State_Cleaned.csv as a new table.')
print('   Relationship: DOSM_HIES_State_Cleaned[State] → DOSM_State_Graduates[State]')
print('   Relationship: DOSM_HIES_State_Cleaned[State] → PRIMARY_Survey_Cleaned[State]')
print()
print('📌 Key Power BI visuals this enables:')
print('   • Choropleth map coloured by CoL_Pressure_Score_Pct')
print('   • Scatter: FreshGrad_as_Pct_StateMedian vs Poverty_Rate_Pct')
print('   • Bar chart: FreshGrad_Expenditure_Coverage_Pct by State (sorted)')
print('   • Matrix: State × Income_Mean vs HIES_Savings_20pct_RM vs Survey savings')

📂 Please upload: hies_state.csv


Saving hies_state.csv to hies_state.csv
✅ Loaded: hies_state.csv — 16 rows × 7 columns

✅ HIES dataset enriched: 16 states × 30 columns
   Zero null values confirmed.

📊 KEY INSIGHTS for FYP:
   Most affordable for fresh grads : Sabah
   Least affordable for fresh grads: W.P. Putrajaya
   Highest CoL pressure            : Kelantan (71.8%)
   Highest poverty rate            : Sabah (19.7%)
   States in Critical CoL zone     : 3
   States affordable for fresh grads: 2 of 16


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


💾 Downloaded: DOSM_HIES_State_Cleaned.csv

📌 Add to Power BI:
   Import DOSM_HIES_State_Cleaned.csv as a new table.
   Relationship: DOSM_HIES_State_Cleaned[State] → DOSM_State_Graduates[State]
   Relationship: DOSM_HIES_State_Cleaned[State] → PRIMARY_Survey_Cleaned[State]

📌 Key Power BI visuals this enables:
   • Choropleth map coloured by CoL_Pressure_Score_Pct
   • Scatter: FreshGrad_as_Pct_StateMedian vs Poverty_Rate_Pct
   • Bar chart: FreshGrad_Expenditure_Coverage_Pct by State (sorted)
   • Matrix: State × Income_Mean vs HIES_Savings_20pct_RM vs Survey savings


---
## 🔗 CELL 16: Build Prescriptive Alerts Table (Separate for Power BI)

In [ ]:
# ── Prescriptive Alerts Table (one row per alert per respondent) ──────────────
# This enables Power BI to display alert details in a dedicated visual

alert_definitions = [
    ('Alert_Needs_Overspend',  'Needs Overspend',
     'Essential spending exceeds 50% of income',
     'Review rent, transport, and loan commitments. Consider cost-sharing.'),
    ('Alert_Wants_Overspend',  'Lifestyle Inflation Alert',
     'Discretionary spending exceeds 30% of income',
     'Track daily wants spending. Cut non-essential subscriptions and dining out.'),
    ('Alert_Savings_Deficit',  'Savings Below Target',
     'Monthly savings fall below 20% of income',
     'Automate savings on pay day. Open a dedicated savings account.'),
    ('Alert_Zero_Savings',     'Critical: Zero Savings',
     'No monthly savings allocation detected',
     'Contact AKPK for free financial counselling. Build a RM 500 emergency buffer first.'),
    ('Alert_DSR_Danger',       'DSR Critical (>60%)',
     'Debt-service ratio exceeds safe threshold of 60%',
     'Seek debt restructuring. Avoid BNPL and new credit cards.'),
]

alert_rows = []
for _, row in df_survey_clean.iterrows():
    for col_flag, alert_name, alert_desc, recommendation in alert_definitions:
        if row[col_flag] == 1:
            alert_rows.append({
                'Respondent_ID':    row['Respondent_ID'],
                'Age_Group':        row['Age_Group'],
                'Gender':           row['Gender'],
                'State':            row['State'],
                'Education_Level':  row['Education_Level'],
                'Income_Band':      row['Monthly_Income_Band'],
                'Alert_Flag':       col_flag,
                'Alert_Name':       alert_name,
                'Alert_Description':alert_desc,
                'Recommendation':   recommendation,
                'Severity':         'Critical' if 'Zero' in alert_name or 'DSR' in alert_name else 'Warning',
                'RAG_Status':       row['RAG_Financial_Status'],
            })

df_alerts = pd.DataFrame(alert_rows)
df_alerts.to_csv('PRESCRIPTIVE_Alerts.csv', index=False, encoding='utf-8-sig')
print(f'✅ Prescriptive Alerts Table: {len(df_alerts)} alert records')
print('💾 Saved: PRESCRIPTIVE_Alerts.csv')
print()
print('📊 Alert Type Distribution:')
print(df_alerts['Alert_Name'].value_counts())

✅ Prescriptive Alerts Table: 129 alert records
💾 Saved: PRESCRIPTIVE_Alerts.csv

📊 Alert Type Distribution:
Alert_Name
Savings Below Target         52
DSR Critical (>60%)          37
Needs Overspend              22
Lifestyle Inflation Alert    18
Name: count, dtype: int64


---
## 🔗 CELL 17: Build HYBRID Dashboard Master Table
> Merges survey data with DOSM benchmarks for respondent vs national comparison

In [ ]:
# ── Merge survey with DOSM 2024 salary benchmarks ────────────────────────────
# Match on Age_Group and Education_Level

dosm_2024 = df_dosm_salary[df_dosm_salary['Year'] == 2024].copy()

# Get fresh grad DOSM benchmark (≤24 / All) for comparison
fg_benchmark_mean   = dosm_2024.loc[dosm_2024['Age_Group']=='24 and Below','Mean_Salary_RM'].values[0]
fg_benchmark_median = dosm_2024.loc[dosm_2024['Age_Group']=='24 and Below','Median_Salary_RM'].values[0]
national_mean       = df_dosm_national[df_dosm_national['Year']==2024]['National_Mean_Salary_RM'].values[0]

df_master = df_survey_clean.copy()
df_master['DOSM_FreshGrad_Mean_2024']       = fg_benchmark_mean
df_master['DOSM_FreshGrad_Median_2024']     = fg_benchmark_median
df_master['DOSM_National_Mean_2024']        = national_mean

# Salary deviation from DOSM benchmark
df_master['Income_vs_DOSM_Deviation_RM']    = (df_master['Income_Midpoint_RM'] - fg_benchmark_mean).round(0)
df_master['Income_vs_DOSM_Deviation_Pct']   = ((df_master['Income_Midpoint_RM'] / fg_benchmark_mean - 1) * 100).round(1)
df_master['Income_Below_DOSM_Benchmark']    = (df_master['Income_Midpoint_RM'] < fg_benchmark_mean).astype(int)

# DOSM 50/30/20 reference thresholds for fresh grads (2024)
df_master['DOSM_Needs_Benchmark_RM']        = round(fg_benchmark_mean * 0.50, 0)
df_master['DOSM_Wants_Benchmark_RM']        = round(fg_benchmark_mean * 0.30, 0)
df_master['DOSM_Savings_Benchmark_RM']      = round(fg_benchmark_mean * 0.20, 0)

# Gap between respondent's savings and DOSM-benchmark savings target
df_master['Savings_vs_DOSM_Gap_RM']         = (df_master['Savings_Midpoint_RM'] - df_master['DOSM_Savings_Benchmark_RM']).round(0)

# Data source label
df_master['Data_Type']    = 'Primary Survey'
df_master['Survey_Year']  = 2026
df_master['DOSM_Ref_Year']= 2024

df_master.to_csv('HYBRID_Dashboard_Master.csv', index=False, encoding='utf-8-sig')
print(f'✅ HYBRID Master Table: {df_master.shape[0]} rows × {df_master.shape[1]} columns')
print('💾 Saved: HYBRID_Dashboard_Master.csv')
print()
print('Sample — DOSM comparison columns (first 5 respondents):')
print(df_master[['Respondent_ID','Income_Midpoint_RM',
                 'DOSM_FreshGrad_Mean_2024',
                 'Income_vs_DOSM_Deviation_RM',
                 'Income_vs_DOSM_Deviation_Pct',
                 'Savings_vs_DOSM_Gap_RM']].head())

✅ HYBRID Master Table: 100 rows × 67 columns
💾 Saved: HYBRID_Dashboard_Master.csv

Sample — DOSM comparison columns (first 5 respondents):
  Respondent_ID  Income_Midpoint_RM  DOSM_FreshGrad_Mean_2024  \
0          R001                1000                      2456   
1          R002                1000                      2456   
2          R003                2000                      2456   
3          R004                1000                      2456   
4          R005                2000                      2456   

   Income_vs_DOSM_Deviation_RM  Income_vs_DOSM_Deviation_Pct  \
0                        -1456                         -59.3   
1                        -1456                         -59.3   
2                         -456                         -18.6   
3                        -1456                         -59.3   
4                         -456                         -18.6   

   Savings_vs_DOSM_Gap_RM  
0                  -341.0  
1                  -341.0  
2

---
## 📈 CELL 18: Summary Statistics & Data Quality Report

In [ ]:

#Summary Statistics & Data Quality Report
print('=' * 70)
print('  FINAL DATA QUALITY & SUMMARY REPORT — FYP HUSNA ZAKARIA')
print('  CRISP-DM Phase 2: Data Preparation Complete')
print('=' * 70)

print(f"\n📊 SURVEY DATASET")
print(f"   Total valid respondents : {len(df_survey_clean)}")
print(f"   Total features          : {df_survey_clean.shape[1]}")
print(f"   Missing values          : {df_survey_clean.isnull().sum().sum()}")

print(f"\n📊 DOSM SALARY BENCHMARKS")
print(f"   Rows                    : {len(df_dosm_salary)}")
print(f"   Years covered           : 2020–2024")

print(f"\n📊 DOSM NATIONAL EMPLOYMENT")
print(f"   Rows                    : {len(df_dosm_national)}")

print(f"\n📊 DOSM STATE GRADUATES")
print(f"   States covered          : {len(df_dosm_state)}")

print(f"\n📊 PRESCRIPTIVE ALERTS")
print(f"   Total alert records     : {len(df_alerts)}")
print(f"   Critical alerts         : {(df_alerts['Severity']=='Critical').sum()}")
print(f"   Warning alerts          : {(df_alerts['Severity']=='Warning').sum()}")

print(f"\n📊 HYBRID MASTER TABLE")
print(f"   Rows × Columns          : {df_master.shape[0]} × {df_master.shape[1]}")

print(f"\n🎯 KEY INSIGHTS (for Power BI dashboard story):")
pct_needs   = df_survey_clean['Alert_Needs_Overspend'].mean()*100
pct_wants   = df_survey_clean['Alert_Wants_Overspend'].mean()*100
pct_savings = df_survey_clean['Alert_Savings_Deficit'].mean()*100
pct_zero    = df_survey_clean['Alert_Zero_Savings'].mean()*100
pct_ka      = df_survey_clean['KA_Gap_Flag'].mean()*100
pct_danger  = (df_survey_clean['RAG_Financial_Status']=='Red — Danger').mean()*100
pct_green   = (df_survey_clean['RAG_Financial_Status']=='Green — On Track').mean()*100

print(f"   Respondents overspending on NEEDS        : {pct_needs:.1f}%")
print(f"   Respondents overspending on WANTS        : {pct_wants:.1f}%")
print(f"   Respondents with savings deficit          : {pct_savings:.1f}%")
print(f"   Respondents with ZERO savings             : {pct_zero:.1f}%")
print(f"   Respondents in Red (Danger) zone          : {pct_danger:.1f}%")
print(f"   Respondents in Green (On Track) zone      : {pct_green:.1f}%")
print(f"   Knowledge-Action Gap rate                 : {pct_ka:.1f}%")
avg_awareness = df_survey_clean['Financial_Awareness_Score'].mean()
print(f"   Avg Financial Awareness Score (1–5)       : {avg_awareness:.2f}")
avg_wants_pct = df_survey_clean['Wants_Pct_of_Income'].mean()
print(f"   Avg Wants % of income (target ≤30%)       : {avg_wants_pct:.1f}%")
avg_savings_pct = df_survey_clean['Savings_Pct_of_Income'].mean()
print(f"   Avg Savings % of income (target ≥20%)     : {avg_savings_pct:.1f}%")

print(f"\n📁 OUTPUT FILES READY FOR POWER BI:")
outputs = [
    ('PRIMARY_Survey_Cleaned.csv',   'Main fact table — survey responses with 50/30/20 logic'),
    ('DOSM_Salary_Benchmarks.csv',   'DOSM benchmark dimension — salary by age & qualification'),
    ('DOSM_Graduate_Employment.csv', 'DOSM trend dimension — national employment 2020–2024'),
    ('DOSM_State_Graduates.csv',     'DOSM state dimension — state-level graduate statistics'),
    ('PRESCRIPTIVE_Alerts.csv',      'Alert fact table — one row per alert per respondent'),
    ('HYBRID_Dashboard_Master.csv',  'Master hybrid table — survey + DOSM merged comparison'),
]
for fname, desc in outputs:
    print(f"   ✅ {fname}")
    print(f"      → {desc}")

print()
print('=' * 70)
print('  Phase 2 COMPLETE ✅ — Ready for Phase 3: Dashboard Development')
print('=' * 70)

  FINAL DATA QUALITY & SUMMARY REPORT — FYP HUSNA ZAKARIA
  CRISP-DM Phase 2: Data Preparation Complete

📊 SURVEY DATASET
   Total valid respondents : 100
   Total features          : 54
   Missing values          : 0

📊 DOSM SALARY BENCHMARKS
   Rows                    : 20
   Years covered           : 2020–2024

📊 DOSM NATIONAL EMPLOYMENT
   Rows                    : 5

📊 DOSM STATE GRADUATES
   States covered          : 16

📊 PRESCRIPTIVE ALERTS
   Total alert records     : 129
   Critical alerts         : 37
   Warning alerts          : 92

📊 HYBRID MASTER TABLE
   Rows × Columns          : 100 × 67

🎯 KEY INSIGHTS (for Power BI dashboard story):
   Respondents overspending on NEEDS        : 22.0%
   Respondents overspending on WANTS        : 18.0%
   Respondents with savings deficit          : 52.0%
   Respondents with ZERO savings             : 0.0%
   Respondents in Red (Danger) zone          : 20.0%
   Respondents in Green (On Track) zone      : 29.0%
   Knowledge-Action Gap ra

---
## 📥 CELL 19: Download All Output Files

In [ ]:
# ── Download all 6 output CSV files to local machine ────────────────────
output_files = [
    'PRIMARY_Survey_Cleaned.csv',
    'DOSM_Salary_Benchmarks.csv',
    'DOSM_Graduate_Employment.csv',
    'DOSM_State_Graduates.csv',
    'PRESCRIPTIVE_Alerts.csv',
    'HYBRID_Dashboard_Master.csv',
]

for f in output_files:
    files.download(f)
    print(f'⬇️  Downloading: {f}')

print()
print('🎓 All files downloaded. Import into Power BI Desktop to build your dashboard.')
print()
print('📌 Power BI Import Order (recommended):')
print('   1. PRIMARY_Survey_Cleaned.csv         → Main Fact Table')
print('   2. DOSM_Graduate_Employment.csv        → Trend Analysis')
print('   3. DOSM_Salary_Benchmarks.csv          → Benchmark Dimension')
print('   4. DOSM_State_Graduates.csv            → Map Visual')
print('   5. PRESCRIPTIVE_Alerts.csv             → Alert Cards')
print('   6. HYBRID_Dashboard_Master.csv         → Comparative Analysis')
print()
print('📌 Key Power BI Relationships:')
print('   PRIMARY_Survey_Cleaned[Respondent_ID] → PRESCRIPTIVE_Alerts[Respondent_ID]')
print('   PRIMARY_Survey_Cleaned[State]         → DOSM_State_Graduates[State]')
print('   HYBRID_Dashboard_Master[Respondent_ID]→ PRESCRIPTIVE_Alerts[Respondent_ID]')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloading: PRIMARY_Survey_Cleaned.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

⬇️  Downloading: PRESCRIPTIVE_Alerts.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloading: HYBRID_Dashboard_Master.csv

🎓 All files downloaded. Import into Power BI Desktop to build your dashboard.

📌 Power BI Import Order (recommended):
   1. PRIMARY_Survey_Cleaned.csv         → Main Fact Table
   2. DOSM_Graduate_Employment.csv        → Trend Analysis
   3. DOSM_Salary_Benchmarks.csv          → Benchmark Dimension
   4. DOSM_State_Graduates.csv            → Map Visual
   5. PRESCRIPTIVE_Alerts.csv             → Alert Cards
   6. HYBRID_Dashboard_Master.csv         → Comparative Analysis

📌 Key Power BI Relationships:
   PRIMARY_Survey_Cleaned[Respondent_ID] → PRESCRIPTIVE_Alerts[Respondent_ID]
   PRIMARY_Survey_Cleaned[State]         → DOSM_State_Graduates[State]
   HYBRID_Dashboard_Master[Respondent_ID]→ PRESCRIPTIVE_Alerts[Respondent_ID]


---
## 📖 CELL 20: Data Dictionary
> Reference table for all engineered columns — include in my FYP documentation

In [ ]:
#Data Dictionary
data_dict = pd.DataFrame([
    # Column, Table, Type, Description, FYP Relevance
    ('Respondent_ID',             'Survey', 'Text',    'Unique respondent identifier (R001–R0xx)',           'Primary key for relationships'),
    ('Age_Group',                 'Survey', 'Text',    '3 bands: 24 and Below | 25-34 | 35 and Above',       'Aligns with DOSM age groups (Jadual 10a)'),
    ('Gender',                    'Survey', 'Text',    'Male | Female',                                      'Demographic segmentation'),
    ('Education_Level',           'Survey', 'Text',    'Degree | Diploma | Masters | PhD',                   'Aligns with DOSM qualification bands'),
    ('State',                     'Survey', 'Text',    'Malaysian state of residence',                       'Links to DOSM_State_Graduates for map visual'),
    ('Area_Type',                 'Survey', 'Text',    'Urban | Rural',                                      'Strata-based analysis (DOSM Jadual 6c)'),
    ('Employment_Status',         'Survey', 'Text',    'Full-Time | Part-Time | Job Seeking | Further Studies', 'User component scope'),
    ('Monthly_Income_Band',       'Survey', 'Text',    'Cleaned income range bands',                         'Input for 50/30/20 engine'),
    ('Income_Midpoint_RM',        'Survey', 'Number',  'Numeric midpoint (RM) of income band',               'Used in all DAX budget threshold calculations'),
    ('Needs_Midpoint_RM',         'Survey', 'Number',  'Estimated actual NEEDS spend (RM midpoint)',         'Compared to Budget_Needs_Limit_RM'),
    ('Wants_Midpoint_RM',         'Survey', 'Number',  'Estimated actual WANTS spend (RM midpoint)',         'Core prescriptive logic variable'),
    ('Savings_Midpoint_RM',       'Survey', 'Number',  'Estimated monthly savings (RM midpoint)',            'Core prescriptive logic variable'),
    ('Budget_Needs_Limit_RM',     'Survey', 'Number',  '50% of income = NEEDS threshold',                    '50/30/20 Rule — Needs limit'),
    ('Budget_Wants_Limit_RM',     'Survey', 'Number',  '30% of income = WANTS threshold',                    '50/30/20 Rule — Wants limit'),
    ('Budget_Savings_Target_RM',  'Survey', 'Number',  '20% of income = SAVINGS target',                     '50/30/20 Rule — Savings target'),
    ('Needs_Gap_RM',              'Survey', 'Number',  'Actual Needs − Budget Needs limit (+ = overspend)', 'Spending gap analysis'),
    ('Wants_Gap_RM',              'Survey', 'Number',  'Actual Wants − Budget Wants limit (+ = overspend)', 'Key lifestyle inflation indicator'),
    ('Savings_Gap_RM',            'Survey', 'Number',  'Savings Target − Actual Savings (+ = shortfall)',   'Savings deficit measure'),
    ('Needs_Pct_of_Income',       'Survey', 'Number',  'Actual NEEDS as % of income',                        'Pie/donut chart in Power BI'),
    ('Wants_Pct_of_Income',       'Survey', 'Number',  'Actual WANTS as % of income',                        'Compared to 30% threshold'),
    ('Savings_Pct_of_Income',     'Survey', 'Number',  'Actual SAVINGS as % of income',                      'Compared to 20% target'),
    ('DSR_Pct',                   'Survey', 'Number',  'Debt Service Ratio (commitments/income × 100)',      'BNM threshold: ≤40% healthy, >60% critical'),
    ('DSR_Status',                'Survey', 'Text',    'Healthy | Warning | Critical',                       'RAG sub-indicator for debt'),
    ('Alert_Needs_Overspend',     'Survey', 'Binary',  '1 = Needs exceeds 50% threshold',                    'Prescriptive Rule 1'),
    ('Alert_Wants_Overspend',     'Survey', 'Binary',  '1 = Lifestyle spending exceeds 30% threshold',       'Prescriptive Rule 2 — core alert'),
    ('Alert_Savings_Deficit',     'Survey', 'Binary',  '1 = Savings below 20% target',                      'Prescriptive Rule 3'),
    ('Alert_Zero_Savings',        'Survey', 'Binary',  '1 = No savings at all (RM 0)',                       'Critical alert — highest risk'),
    ('Alert_DSR_Danger',          'Survey', 'Binary',  '1 = DSR > 60% (BNM danger zone)',                   'Prescriptive Rule 5'),
    ('Alert_Total_Count',         'Survey', 'Number',  'Sum of all alert flags (0–5)',                       'Drives RAG_Financial_Status'),
    ('RAG_Financial_Status',      'Survey', 'Text',    'Green/Amber/Red — financial health status',          'Main visual KPI on dashboard'),
    ('Prescriptive_Advice',       'Survey', 'Text',    'Automated If-Then advice string',                    'Smart Narrative / Card visual in Power BI'),
    ('Financial_Awareness_Score', 'Survey', 'Number',  'Mean of 5 Likert items (1–5 scale)',                 'Objective 1 — awareness analysis'),
    ('Awareness_Level',           'Survey', 'Text',    'High | Moderate | Low Awareness',                    'Segmentation for KA Gap analysis'),
    ('KA_Gap_Flag',               'Survey', 'Binary',  '1 = High awareness BUT savings deficit (KA Gap)',    'Core FYP concept — Knowledge-Action Gap'),
], columns=['Column_Name', 'Table', 'Data_Type', 'Description', 'FYP_Relevance'])

data_dict.to_csv('DATA_DICTIONARY.csv', index=False, encoding='utf-8-sig')
files.download('DATA_DICTIONARY.csv')
print('✅ Data Dictionary exported and downloaded.')
print(f'   Total fields documented: {len(data_dict)}')
print()
print(data_dict[['Column_Name','Data_Type','Description']].to_string(index=False))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Data Dictionary exported and downloaded.
   Total fields documented: 34

              Column_Name Data_Type                                           Description
            Respondent_ID      Text              Unique respondent identifier (R001–R0xx)
                Age_Group      Text          3 bands: 24 and Below | 25-34 | 35 and Above
                   Gender      Text                                         Male | Female
          Education_Level      Text                      Degree | Diploma | Masters | PhD
                    State      Text                          Malaysian state of residence
                Area_Type      Text                                         Urban | Rural
        Employment_Status      Text Full-Time | Part-Time | Job Seeking | Further Studies
      Monthly_Income_Band      Text                            Cleaned income range bands
       Income_Midpoint_RM    Number                  Numeric midpoint (RM) of income band
        Needs_Midpoint_RM